In [113]:
import pandas as pd 
import pymysql 
from getpass import getpass
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error



In [6]:
password = getpass("Enter Password: ")
con = pymysql.connect(
    user = "root",
    host = "localhost",
    database = "bike_store_database",
    password = password
)

cursor = con.cursor()

print("Connected Successfully!!!")


Enter Password:  ········


Connected Successfully!!!


In [8]:
data = pd.read_sql(""" SELECT * FROM store_sale""", con)

C:\Users\Data Professor\AppData\Local\Temp\ipykernel_352\3081336644.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(""" SELECT * FROM store_sale""", con)


In [9]:
print(data)

       puchase_date age       age_group gender         country  \
0        11/26/2013  19     Youth (<25)      M          Canada   
1        11/26/2015  19     Youth (<25)      M          Canada   
2         3/23/2014  49  Adults (35-64)      M       Australia   
3         3/23/2016  49  Adults (35-64)      M       Australia   
4         5/15/2014  47  Adults (35-64)      F       Australia   
...             ...  ..             ...    ...             ...   
113031    4/12/2016  41  Adults (35-64)      M  United Kingdom   
113032     4/2/2014  18     Youth (<25)      M       Australia   
113033     4/2/2016  18     Youth (<25)      M       Australia   
113034     3/4/2014  37  Adults (35-64)      F          France   
113035     3/4/2016  37  Adults (35-64)      F          France   

                   state     category sub_category              product  \
0       British Columbia  Accessories   Bike Racks  Hitch Rack - 4-Bike   
1       British Columbia  Accessories   Bike Racks  Hitch

In [139]:
df = data.copy()

In [140]:
df.columns

Index(['puchase_date', 'age', 'age_group', 'gender', 'country', 'state',
       'category', 'sub_category', 'product', 'quantity', 'unit_cost',
       'unit_price', 'profit', 'cost', 'revenue'],
      dtype='object')

In [141]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113036 entries, 0 to 113035
Data columns (total 15 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   puchase_date  113036 non-null  object
 1   age           113036 non-null  object
 2   age_group     113036 non-null  object
 3   gender        113036 non-null  object
 4   country       113036 non-null  object
 5   state         113036 non-null  object
 6   category      113036 non-null  object
 7   sub_category  113036 non-null  object
 8   product       113036 non-null  object
 9   quantity      113036 non-null  object
 10  unit_cost     113036 non-null  object
 11  unit_price    113036 non-null  object
 12  profit        113036 non-null  object
 13  cost          113036 non-null  object
 14  revenue       113036 non-null  object
dtypes: object(15)
memory usage: 12.9+ MB


In [142]:
df.duplicated().any()

np.True_

In [143]:
df.duplicated().sum()

np.int64(1000)

In [144]:
df.drop_duplicates(inplace = True)

In [145]:
df.isna().any()

puchase_date    False
age             False
age_group       False
gender          False
country         False
state           False
category        False
sub_category    False
product         False
quantity        False
unit_cost       False
unit_price      False
profit          False
cost            False
revenue         False
dtype: bool

In [146]:
df["revenue"] = pd.to_numeric(df["revenue"])
df["cost"] = pd.to_numeric(df["cost"])
df["profit"] = pd.to_numeric(df["profit"])
df["unit_price"] = pd.to_numeric(df["unit_price"])
df["unit_cost"] = pd.to_numeric(df["unit_cost"])
df["quantity"] = pd.to_numeric(df["quantity"])
df["puchase_date"] = pd.to_datetime(df["puchase_date"])

In [147]:
print(df)

       puchase_date age       age_group gender         country  \
0        2013-11-26  19     Youth (<25)      M          Canada   
1        2015-11-26  19     Youth (<25)      M          Canada   
2        2014-03-23  49  Adults (35-64)      M       Australia   
3        2016-03-23  49  Adults (35-64)      M       Australia   
4        2014-05-15  47  Adults (35-64)      F       Australia   
...             ...  ..             ...    ...             ...   
113031   2016-04-12  41  Adults (35-64)      M  United Kingdom   
113032   2014-04-02  18     Youth (<25)      M       Australia   
113033   2016-04-02  18     Youth (<25)      M       Australia   
113034   2014-03-04  37  Adults (35-64)      F          France   
113035   2016-03-04  37  Adults (35-64)      F          France   

                   state     category sub_category              product  \
0       British Columbia  Accessories   Bike Racks  Hitch Rack - 4-Bike   
1       British Columbia  Accessories   Bike Racks  Hitch

In [148]:
x = df.drop(columns=['profit','revenue', 'cost', 'quantity', 'puchase_date', 'age_group'])
y = df['revenue']

In [149]:
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size = 0.25, random_state = 35)


In [150]:
x_train.shape, y_train.shape

((84027, 9), (84027,))

In [151]:
def generate_transformer(predictors):
    predictors = predictors.copy()

    num_cols = predictors.select_dtypes(include=['number']).columns
    cat_cols = predictors.select_dtypes(include=['object']).columns

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy = 'median')),
        ("scaler", StandardScaler())
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy = "most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown = "use_encoded_value", unknown_value = -1))
    ])


    transformer = ColumnTransformer(
        transformers=[
            ("num_pipe", num_pipe, num_cols),
            ("cat_pipe", cat_pipe, cat_cols)
        ],
        remainder = "drop"
    )

    return transformer

processor = generate_transformer(x)

In [152]:
processor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_pipe', ...), ('cat_pipe', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name

In [153]:
def predict_regressor(transformer, model, model_name):

    pipeline = Pipeline([
        ("processor", transformer),
        ("regressor", model)
    ])

    pipeline.fit(x_train, y_train)

    prediction = pipeline.predict(x_test)

    r2 = r2_score(y_test, prediction)
    mae = mean_absolute_error(y_test, prediction)
    mape = mean_absolute_percentage_error(y_test, prediction)

    
    evaluation_table = pd.DataFrame({
        "model_name": [model_name],
        "r2_score": [r2],
        "MAE": [mae],
        "MAPE": [mape]
    })

    return evaluation_table

In [154]:
predict_regressor(processor, LinearRegression(), "Linear Regression")

,model_name,r2_score,MAE,MAPE
0,Linear Regression,0.697787,399.641838,3.239633


In [155]:
predict_regressor(processor, RandomForestRegressor(), "Random Forest")

,model_name,r2_score,MAE,MAPE
0,Random Forest,0.700919,303.293141,1.114024


In [156]:
predict_regressor(processor, DecisionTreeRegressor(), "Decision Tree")

,model_name,r2_score,MAE,MAPE
0,Decision Tree,0.669835,304.678595,1.083227


In [157]:
predict_regressor(processor, XGBRegressor(), "XGB")

,model_name,r2_score,MAE,MAPE
0,XGB,0.718215,313.644318,1.221402


In [159]:
rf_model = Pipeline([
    ("processor", processor),
    ("rf_model", RandomForestRegressor())
]
)
rf_model.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('processor', ...), ('rf_model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_pipe', ...), ('cat_pipe', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [160]:
rf_importance = rf_model.named_steps["rf_model"].feature_importances_

In [161]:
rf_importance

array([0.34316209, 0.48592762, 0.05025947, 0.0085195 , 0.01034194,
       0.03028066, 0.02562707, 0.02206375, 0.0238179 ])

In [162]:
x.columns

Index(['age', 'gender', 'country', 'state', 'category', 'sub_category',
       'product', 'unit_cost', 'unit_price'],
      dtype='object')

In [163]:
importance_df = pd.DataFrame(
    {
        "Features":  x_train.columns,
        "Importance": rf_importance
    }
)

print(importance_df.sort_values(by="Importance",  ascending = False))

       Features  Importance
1        gender    0.485928
0           age    0.343162
2       country    0.050259
5  sub_category    0.030281
6       product    0.025627
8    unit_price    0.023818
7     unit_cost    0.022064
4      category    0.010342
3         state    0.008520


In [168]:
x_df = df.drop(columns=['profit','revenue', 'cost', 'quantity', 'puchase_date', 'age_group'])
y_new = df['revenue']

In [170]:
x_df['price_to_category_avg'] = x_df['unit_price'] / x_df.groupby('category')['unit_price'].transform('mean')
x_df['avg_revenue_country'] = df.groupby('country')['revenue'].transform('mean')
x_df['avg_revenue_product'] = df.groupby('product')['revenue'].transform('mean')
x_df['price_squared'] = df['unit_price'] ** 2
x_df['month'] = pd.to_datetime(df['puchase_date']).dt.month
x_df['day_of_week'] = pd.to_datetime(df['puchase_date']).dt.dayofweek
x_df['price_to_product_avg'] = df['unit_price'] / x_df.groupby('product')['unit_price'].transform('mean')
x_df['price_rank_in_category'] = df.groupby('category')['unit_price'].rank(pct=True)
x_df['product_freq'] = df['product'].map(df['product'].value_counts())
x_df['product_revenue_mean'] = df.groupby('product')['revenue'].transform('mean')

In [171]:
x_train, x_test, y_train, y_test = train_test_split(x_df,y_new, test_size = 0.25, random_state = 35)


In [172]:
x_train.columns

Index(['age', 'gender', 'country', 'state', 'category', 'sub_category',
       'product', 'unit_cost', 'unit_price', 'price_to_category_avg',
       'avg_revenue_country', 'avg_revenue_product', 'price_squared', 'month',
       'day_of_week', 'price_to_product_avg', 'price_rank_in_category',
       'product_freq', 'product_revenue_mean'],
      dtype='object')

In [173]:
print(x_train.head())

      age gender         country             state     category  \
86347  42      M   United States        California  Accessories   
58619  43      M  United Kingdom           England        Bikes   
49809  25      M       Australia   New South Wales        Bikes   
82179  26      M          Canada  British Columbia  Accessories   
95194  21      M   United States        California  Accessories   

          sub_category                 product  unit_cost  unit_price  \
86347  Tires and Tubes     Patch Kit/8 Patches          1           2   
58619       Road Bikes   Road-550-W Yellow, 48        713        1120   
49809   Mountain Bikes  Mountain-200 Black, 38       1252        2295   
82179  Tires and Tubes      Mountain Tire Tube          2           5   
95194  Tires and Tubes      Mountain Tire Tube          2           5   

       price_to_category_avg  avg_revenue_country  avg_revenue_product  \
86347               0.117703           715.167302            27.272621   
58619     

In [179]:
def generate_transformer(predictors):
    predictors = predictors.copy()

    num_cols = predictors.select_dtypes(include=['number']).columns
    cat_cols = predictors.select_dtypes(include=['object']).columns

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy = 'median')),
        ("scaler", StandardScaler())
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy = "most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown = "use_encoded_value", unknown_value = -1))
    ])


    transformer = ColumnTransformer(
        transformers=[
            ("num_pipe", num_pipe, num_cols),
            ("cat_pipe", cat_pipe, cat_cols)
        ],
        remainder = "drop"
    )

    return transformer

processor_new = generate_transformer(x_df)

In [180]:
def predict_regressor(transformer, model, model_name):

    pipeline = Pipeline([
        ("processor", transformer),
        ("regressor", model)
    ])

    pipeline.fit(x_train, y_train)

    prediction = pipeline.predict(x_test)

    r2 = r2_score(y_test, prediction)
    mae = mean_absolute_error(y_test, prediction)
    mape = mean_absolute_percentage_error(y_test, prediction)

    
    evaluation_table = pd.DataFrame({
        "model_name": [model_name],
        "r2_score": [r2],
        "MAE": [mae],
        "MAPE": [mape]
    })

    return evaluation_table

In [181]:
predict_regressor(processor_new, LinearRegression(), "Linear Regression")

,model_name,r2_score,MAE,MAPE
0,Linear Regression,0.715366,346.471939,1.761182


In [182]:
predict_regressor(processor_new, RandomForestRegressor(), "Random Forest")

,model_name,r2_score,MAE,MAPE
0,Random Forest,0.695113,314.622856,1.161328


In [183]:
predict_regressor(processor_new, DecisionTreeRegressor(), "Decision Tree")

,model_name,r2_score,MAE,MAPE
0,Decision Tree,0.46527,351.767787,1.181549


In [184]:
predict_regressor(processor_new, XGBRegressor(), "XGB")

,model_name,r2_score,MAE,MAPE
0,XGB,0.710858,313.205292,1.22677
